# Turkce Wikipedia (trwiki) Ham Dokum -> AI Hazir Parquet Boru Hatti (225 GB Disk Uyumlu)

Bu notebook, resmi Wikimedia sunucularindan en guncel Turkce Wikipedia ham dokumunu indirir,
Wikitext sozluk/sablon kodlarindan arindirir, Zstandard (ZSTD) sikistirmali **Apache Parquet**
formatina donusturur ve 'bitmis paket' olarak dogrudan Google Drive'iniza muhurler.

### 225 GB Disk & Colab Pro Optimizasyonlari:
- **User-Agent Korunmasi:** Wikimedia'nin Python-urllib 403 Forbidden engelini asan kurumsal baslik yapilandirmasi.
- **Cok Cekirdekli Paralellestirme:** `os.cpu_count()` ile tum CPU cekirdekleri (8-12 vCPU) kullanilarak ayiklama suresi 15 dakikadan ~3 dakikaya indirilir.
- **Disk Guvenligi:** Gecici ayiklama alani (`/content/scratch`) 225 GB diskin yalnizca %6'sini kullanir.
- **Drive FUSE Korunmasi:** Google Drive'a **yalnizca temizlenmis nihai Parquet paketi (~1.2 GB)** aktarilir.

In [ ]:
# 1. Adim: Google Drive Baglantisi
from google.colab import drive
import os

drive.mount('/content/drive')
print('[OK] Google Drive basariyla baglandi.')

In [ ]:
# 2. Adim: Gerekli Araclarin Kurulumu
!apt-get update -qq && apt-get install -y -qq aria2
!pip install -q wikiextractor pyarrow tqdm
print('[OK] Sistem araclari ve Python kutuphaneleri hazir.')

In [ ]:
# 3. Adim: Dizin ve Calisma Yapilandirmasi
LANG = "tr"
LOCAL_SCRATCH = "/content/scratch"
EXTRACT_DIR = os.path.join(LOCAL_SCRATCH, "extracted")
DRIVE_TARGET = "/content/drive/MyDrive/Wikipedia_Datasets/tr"
DRIVE_DATA_DIR = os.path.join(DRIVE_TARGET, "data")
DRIVE_RAW_DIR = os.path.join(DRIVE_TARGET, "raw")

# Ham bz2 dosyasini da Drive'a yedeklemek ister misiniz?
BACKUP_RAW_DUMP = True

os.makedirs(LOCAL_SCRATCH, exist_ok=True)
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
if BACKUP_RAW_DUMP:
    os.makedirs(DRIVE_RAW_DIR, exist_ok=True)

print(f"[YAPILANDIRMA] Dil: {LANG}")
print(f"[YAPILANDIRMA] Gecici Alan: {LOCAL_SCRATCH}")
print(f"[YAPILANDIRMA] Google Drive Hedefi: {DRIVE_TARGET}")

In [ ]:
# 4. Adim: Hizli Indirme ve MD5 Dogrulamasi (User-Agent Uyumlu)
import subprocess
import hashlib
import urllib.request

DUMP_FILE = f"{LANG}wiki-latest-pages-articles.xml.bz2"
DUMP_URL = f"https://dumps.wikimedia.org/{LANG}wiki/latest/{DUMP_FILE}"
MD5_URL = f"https://dumps.wikimedia.org/{LANG}wiki/latest/{LANG}wiki-latest-md5sums.txt"
LOCAL_DUMP_PATH = os.path.join(LOCAL_SCRATCH, DUMP_FILE)
LOCAL_MD5_PATH = os.path.join(LOCAL_SCRATCH, "md5sums.txt")

# Wikimedia politikasi geregi tanimli User-Agent zorunludur (403 Forbidden engellemesi)
USER_AGENT = "WikipediaDatasetPipeline/1.0 (Academic Research; contact@dataset-pipeline.local)"

print(f"[1/4] MD5 kontrol listesi indiriliyor...")
req = urllib.request.Request(MD5_URL, headers={"User-Agent": USER_AGENT})
with urllib.request.urlopen(req) as resp, open(LOCAL_MD5_PATH, "wb") as f:
    f.write(resp.read())

expected_md5 = None
with open(LOCAL_MD5_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 2 and parts[1].replace('*', '') == DUMP_FILE:
            expected_md5 = parts[0]
            break

print(f"[META] Beklenen MD5: {expected_md5}")
print(f"[2/4] aria2c ile yuksek hizli indirme baslatiliyor: {DUMP_URL}")
cmd = [
    "aria2c",
    f"--user-agent={USER_AGENT}",
    "-x", "8", "-s", "8", "-j", "4", "-c",
    "-d", LOCAL_SCRATCH, "-o", DUMP_FILE, DUMP_URL
]
subprocess.run(cmd, check=True)

print(f"[3/4] Yerel MD5 dogrulaniyor...")
hasher = hashlib.md5()
with open(LOCAL_DUMP_PATH, 'rb') as f:
    while chunk := f.read(1024 * 1024 * 8):
        hasher.update(chunk)
actual_md5 = hasher.hexdigest()

if expected_md5 and actual_md5 != expected_md5:
    raise ValueError(f"MD5 Dogrulama Hatasi! Beklenen: {expected_md5}, Hesaplanan: {actual_md5}")

print(f"[OK] MD5 dogrulandi: {actual_md5}")

# Istege bagli ham yedek Drive'a kopyalama
if BACKUP_RAW_DUMP:
    print(f"[YEDEK] Ham bz2 dosyasi Drive'a kopyalaniyor...")
    import shutil
    shutil.copy2(LOCAL_DUMP_PATH, os.path.join(DRIVE_RAW_DIR, DUMP_FILE))
    shutil.copy2(LOCAL_MD5_PATH, os.path.join(DRIVE_RAW_DIR, f"{LANG}wiki-latest-md5sums.txt"))
    print("[OK] Ham dosya Drive raw/ dizinine yedeklendi.")

In [ ]:
# 5. Adim: Cok Cekirdekli Hizli Wikitext Ayiklama
import subprocess
import os

cpu_count = os.cpu_count() or 4
print(f"[AYRISTIRMA] wikiextractor {cpu_count} paralel CPU cekirdegi ile baslatiliyor...")
print("Colab Pro coklu CPU destegiyle bu islem yaklasik 2-4 dakika surecektir.")

# --json: JSONL formatinda cikti verir
# --no-templates: Sablon kodlarini ayiklar
# --processes: Cok cekirdekli paralellestirme
# -b 100M: Parca boyutu
cmd_extract = [
    "python", "-m", "wikiextractor.WikiExtractor",
    LOCAL_DUMP_PATH,
    "--json",
    "--no-templates",
    "--processes", str(cpu_count),
    "-b", "100M",
    "-o", EXTRACT_DIR
]
subprocess.run(cmd_extract, check=True)
print("[OK] Cok cekirdekli Wikitext ayiklama tamamlandi.")

In [ ]:
# 6. Adim: Parquet + Zstandard (ZSTD) Donusturme ve Drive'a Muhurleme
import os
import json
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

print("[PARQUET] JSON dosyalari taranarak Parquet (ZSTD) formatina paketleniyor...")

schema = pa.schema([
    ('id', pa.string()),
    ('url', pa.string()),
    ('title', pa.string()),
    ('text', pa.string())
])

all_files = []
for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        if file.startswith("wiki_"):
            all_files.append(os.path.join(root, file))

all_files.sort()
print(f"Toplam {len(all_files)} adet cikarilmis blok bulundu.")

CHUNK_SIZE = 100000  # Her 100.000 makalede bir ayri Parquet dosyasi olusturulur
current_records = {'id': [], 'url': [], 'title': [], 'text': []}
part_index = 0
total_articles = 0
total_chars = 0

def write_parquet_part(records, p_index):
    out_name = f"{LANG}wiki-part-{p_index:05d}.parquet"
    out_path = os.path.join(DRIVE_DATA_DIR, out_name)
    table = pa.Table.from_pydict(records, schema=schema)
    pq.write_table(table, out_path, compression='zstd', compression_level=6)
    size_mb = os.path.getsize(out_path) / (1024 * 1024)
    print(f"[KAYIT] {out_name} olusturuldu ({len(records['id'])} makale, {size_mb:.2f} MB)")

for f_path in tqdm(all_files, desc="Isleniyor"):
    with open(f_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                txt = obj.get('text', '').strip()
                title = obj.get('title', '').strip()
                if not txt or not title:
                    continue
                
                current_records['id'].append(str(obj.get('id', '')))
                current_records['url'].append(str(obj.get('url', '')))
                current_records['title'].append(title)
                current_records['text'].append(txt)
                
                total_articles += 1
                total_chars += len(txt)
                
                if len(current_records['id']) >= CHUNK_SIZE:
                    write_parquet_part(current_records, part_index)
                    part_index += 1
                    current_records = {'id': [], 'url': [], 'title': [], 'text': []}
            except Exception:
                continue

if len(current_records['id']) > 0:
    write_parquet_part(current_records, part_index)
    part_index += 1

print(f"\n[TAMAMLANDI] Toplam Makale Sayisi: {total_articles:,}")
print(f"[TAMAMLANDI] Toplam Karakter: {total_chars:,}")
print(f"[TAMAMLANDI] Tahmini Token (Tr ~0.3): ~{int(total_chars * 0.3):,} token")

In [ ]:
# 7. Adim: Manifest Karnesini Uretme ve Kaydetme
import datetime

manifest = {
    "dataset": "wikipedia",
    "language": LANG,
    "source": "dumps.wikimedia.org",
    "dump_file": DUMP_FILE,
    "md5": actual_md5,
    "format": "parquet",
    "compression": "zstd-6",
    "schema": ["id", "url", "title", "text"],
    "article_count": total_articles,
    "character_count": total_chars,
    "estimated_tokens": int(total_chars * 0.3),
    "parts_count": part_index,
    "created_at": datetime.datetime.now(datetime.timezone.utc).isoformat()
}

manifest_path = os.path.join(DRIVE_TARGET, "manifest.json")
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f"[MANIFEST] Karne kaydedildi: {manifest_path}")

In [ ]:
# 8. Adim: Nihai Dogrulama (Ornek Makale Gosterimi)
import pyarrow.parquet as pq

first_part = os.path.join(DRIVE_DATA_DIR, f"{LANG}wiki-part-00000.parquet")
table = pq.read_table(first_part)
df = table.slice(0, 3).to_pandas()

print("=== ILK 3 ORNEK MAKALE ===\n")
for idx, row in df.iterrows():
    print(f"--- [{row['id']}] {row['title']} ---")
    print(f"URL: {row['url']}")
    snippet = row['text'][:250].replace('\n', ' ')
    print(f"Metin: {snippet}...\n")

print("[BASARILI] Turkce Wikipedia veri seti Google Drive'a tam paket olarak muhurlendi.")

In [ ]:
# 9. Adim: Gecici Calisma Alanini Temizleme (Istege Bagli)
import shutil

# Parquet dosyalari artik Google Drive'da guvende oldugu icin
# Colab yerel diskindeki ara JSON bloklarini silebilirsiniz.
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
    print('[TEMIZLIK] Gecici JSON dosyalari silindi, yerel disk alani temizlendi.')